In [ ]:
!uv pip install -q torch torchvision matplotlib numpy

# Module 2: Deep Learning Primer — Image Classification with PyTorch

**Module 02 | Notebook 2 of 2**

## Introduction

In `01_neural_network_basics.ipynb` you built an MNIST classifier with TensorFlow/Keras.
This notebook builds the **exact same classifier** in **PyTorch** — same dataset, same
architecture (784 → 128 → 64 → 10), same evaluation metric. By the end, you will
understand what is the same and what is different between the two frameworks.

### What You Will Learn

- PyTorch core objects: `Tensor`, `Dataset`, `DataLoader`, `nn.Module`
- The explicit training loop: `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`
- Why the explicit loop matters: you will use this pattern directly in Module 07 (HuggingFace fine-tuning)
- How to evaluate a PyTorch model and visualize predictions

### Prerequisites

Complete `01_neural_network_basics.ipynb` first — this notebook is a direct comparison

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

# Check PyTorch version and device
print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print()
print("Compare: TF/Keras loaded MNIST as numpy arrays directly.")
print("PyTorch wraps datasets in Dataset + DataLoader — the same")
print("abstraction that HuggingFace Trainer uses for text data in Module 07.")

## 1. Loading Data with DataLoader

PyTorch separates the concept of **what** data is (a `Dataset`) from **how** it is loaded (a `DataLoader`).

- `Dataset`: knows how to get one sample at a time
- `DataLoader`: batches samples, shuffles, and can load in parallel

In `01_neural_network_basics.ipynb`, we loaded MNIST as a numpy array and passed it directly to `model.fit()`.
Here, the DataLoader handles batching automatically during the training loop.

In [ ]:
# Define transforms: ToTensor converts PIL images to float tensors, Normalize standardizes
transform = transforms.Compose([
    transforms.ToTensor(),           # PIL image (0-255) → float tensor (0.0-1.0)
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std (pre-computed)
])

# Download and load MNIST
train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader: handles batching, shuffling, and parallel loading
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# Peek at one batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")   # [64, 1, 28, 28] — channels first in PyTorch
print(f"Label shape: {labels.shape}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {labels[i].item()}', fontsize=11)
    ax.axis('off')
plt.suptitle('Sample MNIST Images (PyTorch DataLoader)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Defining the Model with nn.Module

In TensorFlow/Keras, you call `keras.Sequential([layers...])`.
In PyTorch, you subclass `nn.Module` and define `forward()`.

The `forward()` method is called automatically when you do `output = model(input)`.
This explicit pattern gives you full control over the computation graph —
which is why HuggingFace models (Transformers, LoRA adapters) are all `nn.Module` subclasses.

In [ ]:
class MNISTClassifier(nn.Module):
    """
    Same architecture as NB01 (TF/Keras):
    784 → 128 → ReLU → 64 → ReLU → 10
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()      # [batch, 1, 28, 28] → [batch, 784]
        self.fc1     = nn.Linear(784, 128)
        self.fc2     = nn.Linear(128, 64)
        self.fc3     = nn.Linear(64, 10)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)                  # Raw logits — CrossEntropyLoss expects these (no softmax here)
        return x

model = MNISTClassifier().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print("(Same as NB01: ~109,000 parameters)")

## 3. The Explicit Training Loop

**Keras:** `model.fit(X, y, epochs=10)` — the framework runs the loop for you.

**PyTorch:** you write the loop yourself. Every step is explicit:
1. `optimizer.zero_grad()` — clear gradients from the last batch (they accumulate by default)
2. `output = model(images)` — forward pass
3. `loss = criterion(output, labels)` — compute loss
4. `loss.backward()` — backpropagate: compute gradients for all parameters
5. `optimizer.step()` — update parameters using the gradients

This explicitness is why PyTorch is preferred for research — you can modify any step.
In Module 07, HuggingFace Trainer runs this same loop internally, with gradient
accumulation, mixed precision, and checkpointing added on top.

In [ ]:
# Loss function and optimizer — same choices as NB01
criterion = nn.CrossEntropyLoss()            # Combines log-softmax + NLLLoss; expects raw logits
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses    = []
val_accuracies  = []

EPOCHS = 10
print(f"Training for {EPOCHS} epochs on {device}...\n")

for epoch in range(EPOCHS):
    # --- Training phase ---
    model.train()          # Enable dropout (training mode)
    epoch_loss = 0.0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()                       # 1. Clear gradients
        outputs = model(batch_images)               # 2. Forward pass
        loss = criterion(outputs, batch_labels)     # 3. Compute loss
        loss.backward()                             # 4. Backward pass
        optimizer.step()                            # 5. Update parameters

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- Validation phase ---
    model.eval()           # Disable dropout (eval mode)
    correct = total = 0
    with torch.no_grad():  # No gradient computation needed for evaluation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            _, preds = outputs.max(1)   # argmax — class with highest logit
            total   += labels.size(0)
            correct += preds.eq(labels).sum().item()

    val_acc = correct / total
    val_accuracies.append(val_acc)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.4f}")

print("\n✅ Training complete!")

## 4. Evaluation

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs  = model(images)
        _, preds = outputs.max(1)
        total   += labels.size(0)
        correct += preds.eq(labels).sum().item()

test_accuracy = correct / total
print(f"Final Test Accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")
print()
print("Compare with NB01 (TensorFlow/Keras): ~97-98%")
print("Both frameworks, same architecture → same result. The framework is a tool, not the answer.")

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, EPOCHS+1), train_losses, 'b-o', linewidth=2, markersize=5)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS+1), [v*100 for v in val_accuracies], 'g-o', linewidth=2, markersize=5)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Validation Accuracy Over Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize predictions
images_batch, labels_batch = next(iter(test_loader))
model.eval()
with torch.no_grad():
    outputs_batch = model(images_batch.to(device))
    _, preds_batch = outputs_batch.max(1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images_batch[i].squeeze(), cmap='gray')
    pred  = preds_batch[i].item()
    true  = labels_batch[i].item()
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Pred: {pred} | True: {true}', color=color, fontsize=10)
    ax.axis('off')
plt.suptitle('Test Predictions (green = correct, red = wrong)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. TensorFlow/Keras vs. PyTorch — Side by Side

| Step | TensorFlow/Keras (NB01) | PyTorch (this notebook) |
|------|------------------------|------------------------|
| **Data loading** | numpy arrays passed to `model.fit()` | `Dataset` + `DataLoader` |
| **Model definition** | `keras.Sequential([layers...])` | `class Model(nn.Module): def forward(x)` |
| **Training** | `model.fit(X, y, epochs=N)` | Explicit loop: zero_grad → forward → backward → step |
| **Loss** | string `'sparse_categorical_crossentropy'` | `nn.CrossEntropyLoss()` object |
| **Optimizer** | string `'adam'` | `optim.Adam(model.parameters())` |
| **Eval mode** | automatic | `model.eval()` + `torch.no_grad()` |
| **Prediction** | `model.predict(X)` | `outputs.max(1)` on raw logits |

**Which should you use?**
- **TensorFlow/Keras:** Great for rapid prototyping, production serving (TF Serving, Vertex AI)
- **PyTorch:** Dominant in research; the entire HuggingFace ecosystem (Transformers, PEFT, Diffusers) is PyTorch-native

In **Module 07 (fine-tuning)**, you will use HuggingFace `Trainer` — which is built on PyTorch `nn.Module`
and the same `DataLoader` abstraction you used in this notebook.

## 6. Student Exercise

**Challenge:** Improve this PyTorch classifier.

Try any of:
- Add more hidden layers (e.g. 784 → 256 → 128 → 64 → 10)
- Replace `nn.ReLU()` with `nn.GELU()` — the activation used in GPT-2 and BERT
- Add `nn.BatchNorm1d(n)` after each `nn.Linear` layer
- Try `lr=0.01` and observe training stability vs. `lr=0.001`
- Add `weight_decay=1e-4` to `optim.Adam` (L2 regularization)

Expected accuracy with improvements: **98-99%** on MNIST test set.

In [ ]:
# Student Exercise: Build an improved PyTorch classifier

class ImprovedMNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        # TODO: Add your layers here
        # Example:
        # self.fc1  = nn.Linear(784, 256)
        # self.bn1  = nn.BatchNorm1d(256)
        # self.act  = nn.GELU()    # nn.GELU: activation used in GPT-2, BERT, modern LLMs
        # self.drop = nn.Dropout(0.3)

    def forward(self, x):
        x = self.flatten(x)
        # TODO: implement forward pass
        return x

# TODO: instantiate, train, and evaluate
# my_improved_model = ImprovedMNISTClassifier().to(device)
# optimizer_improved = optim.Adam(my_improved_model.parameters(), lr=0.001)
# ... same training loop as above ...
# Expected: ~98-99% test accuracy

## Summary

In this notebook, you built the same MNIST classifier as NB01 — but in PyTorch:

✅ **DataLoader** — the same abstraction HuggingFace Trainer uses for text data (Module 07)
✅ **nn.Module** — the base class for every HuggingFace model (Transformers, LoRA adapters)
✅ **Explicit training loop** — `zero_grad → forward → backward → step` — the engine behind all PyTorch training
✅ **model.train() / model.eval()** — controls dropout and batch norm behavior

### Next Step

**Module 03: Overview of Generative AI** — you will build a Variational Autoencoder (VAE)
whose encoder and decoder are neural networks (like this one) applied to image compression,
then explore how diffusion models use these latent representations to generate images from text.

### Resources

- [PyTorch Tutorials](https://pytorch.org/tutorials/)
- [HuggingFace Transformers (PyTorch-based)](https://huggingface.co/docs/transformers)
- [nn.Module documentation](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)